In [1]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
!pip install -q catboost scikit-learn pandas numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python3.14 -m pip install --upgrade pip


In [2]:
# ── Cell 2: Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report

SEEDS    = [42, 7, 123]
N_SPLITS = 10
print('Libraries loaded.')

Libraries loaded.


In [3]:
# ── Cell 3: Load data ────────────────────────────────────────────────────────
TRAIN_DATA  = pd.read_csv('train-data.csv',  index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA   = pd.read_csv('test-data.csv',   index_col='id')

print(f'Train: {TRAIN_DATA.shape}, Test: {TEST_DATA.shape}')
print(TRAIN_LABEL['disorder'].value_counts().sort_index())

Train: (13249, 41), Test: (8834, 41)
disorder
0     389
1    2068
2    1090
3    3096
4      58
5    1700
6     813
7    2643
8      91
9    1301
Name: count, dtype: int64


In [4]:
# ── Cell 4: Preprocessing ────────────────────────────────────────────────────
# a3 base features +
# NEW: coordinate features from institute_location (from 26th-place solution)
# NEW: ratio features age/mother_age, age/father_age, wbcc/bcc etc.
# NEW: log transforms on numeric cols

def extract_coords(loc_str):
    """Extract (lat, lon) from institute_location string. Returns (NaN, NaN) if not found."""
    if pd.isna(loc_str) or loc_str == '-':
        return np.nan, np.nan
    match = re.search(r'\(([\-\d.]+),\s*([\-\d.]+)\)', str(loc_str))
    if match:
        return float(match.group(1)), float(match.group(2))
    return np.nan, np.nan


def preprocess(df):
    df = df.copy()

    # ── Coordinate features BEFORE dropping institute_location ────────────────
    coords = df['institute_location'].apply(extract_coords)
    df['lat'] = coords.apply(lambda x: x[0])
    df['lon'] = coords.apply(lambda x: x[1])

    # Convert lat/lon to 3D unit-sphere coordinates (handles the circular nature
    # of longitude — -180 and +180 are the same place)
    lat_rad = np.deg2rad(df['lat'].fillna(0))
    lon_rad = np.deg2rad(df['lon'].fillna(0))
    df['coord_x'] = np.cos(lat_rad) * np.cos(lon_rad)
    df['coord_y'] = np.cos(lat_rad) * np.sin(lon_rad)
    df['coord_z'] = np.sin(lat_rad)
    # Flag for rows where coordinate was missing
    df['coord_missing'] = df['lat'].isna().astype(int)
    df = df.drop(columns=['lat', 'lon'])

    # ── Drop zero-signal columns ──────────────────────────────────────────────
    drop_cols = [
        'first_name', 'last_name', 'insitute_name', 'institute_location',
        'test_1', 'test_2', 'test_3', 'test_4', 'test_5', 'treatment_consent'
    ]
    df = df.drop(columns=drop_cols)

    # ── Missing flags BEFORE encoding ─────────────────────────────────────────
    miss_cols = [
        'gender', 'maternal_defect', 'mother_age', 'father_age',
        'respiration', 'heart_rate', 'risk_level', 'place_birth',
        'folic_acid', 'maternal_illness', 'infertility_treatment',
        'problem_previous_pregnancies', 'abortion_cnt',
        'birth_defects', 'white_blood_cell_count', 'blood_test',
        'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5'
    ]
    df['missing_count']      = df[miss_cols].isna().sum(axis=1)
    df['missing_parent_age'] = df['mother_age'].isna().astype(int) + df['father_age'].isna().astype(int)
    df['missing_symptoms']   = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].isna().sum(axis=1)
    df['missing_clinical']   = df[['respiration','heart_rate','risk_level','blood_test']].isna().sum(axis=1)

    for col in ['mother_age','father_age','maternal_defect','gender',
                'risk_level','heart_rate','respiration','abortion_cnt','white_blood_cell_count']:
        df[f'{col}_missing'] = df[col].isna().astype(int)

    # ── Encode ────────────────────────────────────────────────────────────────
    binary_yn = [
        'mother_defect','father_defect','maternal_defect','paternal_defect',
        'alive','folic_acid','maternal_illness','infertility_treatment',
        'problem_previous_pregnancies',
        'symptom_1','symptom_2','symptom_3','symptom_4','symptom_5'
    ]
    for col in binary_yn:
        df[col] = df[col].map({'Y': 1, 'N': 0})

    df['respiration']   = df['respiration'].map({'A': 1, 'N': 0})
    df['heart_rate']    = df['heart_rate'].map({'A': 1, 'N': 0})
    df['risk_level']    = df['risk_level'].map({'H': 1, 'L': 0})
    df['place_birth']   = df['place_birth'].map({'I': 1, 'H': 0})
    df['birth_defects'] = df['birth_defects'].map({'S': 1, 'M': 2})
    df['gender']        = df['gender'].map({'M': 0, 'F': 1, 'A': 2})
    df['autopsy']       = df['autopsy'].map({'Y': 1, 'N': 0})
    df['blood_test']    = df['blood_test'].map({'N': 0, 'I': 1, 'S': 2, 'A': 3})

    for col in ['birth_asphyxia', 'radiation_exposure', 'substance_abuse']:
        df[col] = df[col].map({'Y': 1, 'N': 0, 'NR': 2})

    # ── Impute numerics for ratio/log features ────────────────────────────────
    for col in ['age', 'mother_age', 'father_age', 'blood_cell_count', 'white_blood_cell_count', 'abortion_cnt']:
        df[col] = df[col].fillna(df[col].median())

    # ── NEW: Ratio features (from 26th-place HackerEarth solution) ────────────
    df['age_per_mom']   = df['age'] / (df['mother_age'] + 1)
    df['age_per_dad']   = df['age'] / (df['father_age'] + 1)
    df['age_per_bcc']   = df['age'] / (df['blood_cell_count'] + 1e-5)
    df['age_per_wbcc']  = df['age'] / (df['white_blood_cell_count'] + 1e-5)
    df['wbcc_per_bcc']  = df['white_blood_cell_count'] / (df['blood_cell_count'] + 1e-5)
    df['parent_age_sum']= df['mother_age'] + df['father_age']

    # ── NEW: Log transforms on skewed numerics ────────────────────────────────
    for col in ['age', 'blood_cell_count', 'mother_age', 'father_age', 'white_blood_cell_count']:
        df[f'log_{col}'] = np.log1p(df[col])

    # ── a3 engineered features ────────────────────────────────────────────────
    df['defect_sum']  = df[['mother_defect','father_defect','maternal_defect','paternal_defect']].sum(axis=1)
    df['symptom_sum'] = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].sum(axis=1)

    df['defect_x_symptom']     = df['defect_sum'] * df['symptom_sum']
    df['any_defect']           = (df['defect_sum'] > 0).astype(int)
    df['any_symptom']          = (df['symptom_sum'] > 0).astype(int)
    df['high_symptom']         = (df['symptom_sum'] >= 4).astype(int)
    df['all_defects']          = (df['defect_sum'] == 4).astype(int)
    df['parent_age_gap']       = (df['father_age'] - df['mother_age']).abs()
    df['symptom_defect_ratio'] = df['symptom_sum'] / (df['defect_sum'] + 1)

    df['s4_and_s5']    = ((df['symptom_4'] == 1) & (df['symptom_5'] == 1)).astype(int)
    df['no_s4_s5']     = ((df['symptom_4'] == 0) & (df['symptom_5'] == 0)).astype(int)
    df['late_vs_early']= (df['symptom_4'].fillna(0) + df['symptom_5'].fillna(0)
                         - df['symptom_1'].fillna(0) - df['symptom_2'].fillna(0))
    df['weighted_sym'] = (df['symptom_1'].fillna(0)*1 + df['symptom_2'].fillna(0)*1 +
                          df['symptom_3'].fillna(0)*1 + df['symptom_4'].fillna(0)*2 +
                          df['symptom_5'].fillna(0)*2)

    df['both_parents_defect'] = ((df['mother_defect'] == 1) & (df['father_defect'] == 1)).astype(int)
    df['no_parent_defect']    = ((df['mother_defect'] == 0) & (df['father_defect'] == 0)).astype(int)

    return df


X_train_base = preprocess(TRAIN_DATA)
X_test_base  = preprocess(TEST_DATA)
y_train      = TRAIN_LABEL['disorder'].values

# Fill any remaining NaNs for sklearn models (ET/RF don't handle NaN)
X_train_filled = X_train_base.fillna(-999)
X_test_filled  = X_test_base.fillna(-999)

print(f'Base features: {X_train_base.shape[1]}')
print(f'X_train: {X_train_base.shape}, X_test: {X_test_base.shape}')

Base features: 74
X_train: (13249, 74), X_test: (8834, 74)


In [5]:
# ── Cell 5: ETRF — generate OOF meta-features ────────────────────────────────
# From the PMC paper (Raza et al. 2023):
#   Train ExtraTree + RandomForest with OOF cross-validation
#   Append their class probability outputs as new features
#   This gives CatBoost a "pre-digested" view of the data from 2 different algorithms
#
# IMPORTANT: we use OOF probabilities for train, full-fit probabilities for test
# This prevents leakage — the meta-features for each train row come from
# a model that never saw that row during training

print('Generating ETRF meta-features...')

N_CLASSES = 10
skf_meta  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ET OOF
et_oof_train  = np.zeros((len(y_train), N_CLASSES))
et_pred_test  = np.zeros((len(X_test_filled), N_CLASSES))

# RF OOF
rf_oof_train  = np.zeros((len(y_train), N_CLASSES))
rf_pred_test  = np.zeros((len(X_test_filled), N_CLASSES))

for fold, (tr_idx, val_idx) in enumerate(skf_meta.split(X_train_filled, y_train)):
    X_tr = X_train_filled.iloc[tr_idx].values
    X_val= X_train_filled.iloc[val_idx].values
    y_tr = y_train[tr_idx]

    # ExtraTreesClassifier
    et = ExtraTreesClassifier(
        n_estimators  = 500,
        class_weight  = 'balanced',
        random_state  = 42,
        n_jobs        = -1,
    )
    et.fit(X_tr, y_tr)
    et_oof_train[val_idx] = et.predict_proba(X_val)
    et_pred_test          += et.predict_proba(X_test_filled.values) / skf_meta.n_splits

    # RandomForestClassifier
    rf = RandomForestClassifier(
        n_estimators  = 500,
        class_weight  = 'balanced',
        random_state  = 42,
        n_jobs        = -1,
    )
    rf.fit(X_tr, y_tr)
    rf_oof_train[val_idx] = rf.predict_proba(X_val)
    rf_pred_test          += rf.predict_proba(X_test_filled.values) / skf_meta.n_splits

    et_ba = balanced_accuracy_score(y_train[val_idx], np.argmax(et_oof_train[val_idx], axis=1))
    rf_ba = balanced_accuracy_score(y_train[val_idx], np.argmax(rf_oof_train[val_idx], axis=1))
    print(f'  Fold {fold+1}: ET BA={et_ba:.4f}  RF BA={rf_ba:.4f}')

# ET/RF standalone OOF scores
et_oof_ba = balanced_accuracy_score(y_train, np.argmax(et_oof_train, axis=1))
rf_oof_ba = balanced_accuracy_score(y_train, np.argmax(rf_oof_train, axis=1))
print(f'\nET  OOF BA: {et_oof_ba:.4f}')
print(f'RF  OOF BA: {rf_oof_ba:.4f}')

# ── Append meta-features to base features ────────────────────────────────────
et_cols = [f'et_p{i}' for i in range(N_CLASSES)]
rf_cols = [f'rf_p{i}' for i in range(N_CLASSES)]

X_train_meta = pd.concat([
    X_train_base.reset_index(drop=True),
    pd.DataFrame(et_oof_train, columns=et_cols),
    pd.DataFrame(rf_oof_train, columns=rf_cols),
], axis=1)

X_test_meta = pd.concat([
    X_test_base.reset_index(drop=True),
    pd.DataFrame(et_pred_test,  columns=et_cols),
    pd.DataFrame(rf_pred_test,  columns=rf_cols),
], axis=1)

print(f'\nFinal feature count: {X_train_meta.shape[1]}  (+20 ETRF meta-features)')

Generating ETRF meta-features...
  Fold 1: ET BA=0.2499  RF BA=0.2314
  Fold 2: ET BA=0.2461  RF BA=0.2425
  Fold 3: ET BA=0.2516  RF BA=0.2433
  Fold 4: ET BA=0.2471  RF BA=0.2341
  Fold 5: ET BA=0.2850  RF BA=0.2573

ET  OOF BA: 0.2558
RF  OOF BA: 0.2414

Final feature count: 94  (+20 ETRF meta-features)


In [6]:
# ── Cell 6: Class weights ────────────────────────────────────────────────────
class_counts  = np.bincount(y_train)
class_weights = len(y_train) / (10 * class_counts)

print('Class weights:')
for i, (n, w) in enumerate(zip(class_counts, class_weights)):
    print(f'  Class {i}: weight={w:.3f}  (n={n})')

Class weights:
  Class 0: weight=3.406  (n=389)
  Class 1: weight=0.641  (n=2068)
  Class 2: weight=1.216  (n=1090)
  Class 3: weight=0.428  (n=3096)
  Class 4: weight=22.843  (n=58)
  Class 5: weight=0.779  (n=1700)
  Class 6: weight=1.630  (n=813)
  Class 7: weight=0.501  (n=2643)
  Class 8: weight=14.559  (n=91)
  Class 9: weight=1.018  (n=1301)


In [7]:
# ── Cell 7: CatBoost on enriched features — 3 seeds x 10 folds ──────────────
all_oof_proba  = np.zeros((len(y_train), 10))
all_test_preds = np.zeros((len(X_test_meta), 10))

for SEED in SEEDS:
    print(f"\n{'='*40} SEED={SEED} {'='*40}")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    oof_proba   = np.zeros((len(y_train), 10))
    test_preds  = np.zeros((len(X_test_meta), 10))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_meta, y_train)):
        X_tr,  X_val = X_train_meta.iloc[tr_idx], X_train_meta.iloc[val_idx]
        y_tr,  y_val = y_train[tr_idx],           y_train[val_idx]

        model = CatBoostClassifier(
            iterations            = 2000,
            learning_rate         = 0.03,
            depth                 = 6,
            l2_leaf_reg           = 3,
            class_weights         = class_weights,
            early_stopping_rounds = 100,
            eval_metric           = 'Accuracy',
            random_seed           = SEED,
            verbose               = 0,
            thread_count          = -1,
        )
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

        val_proba = model.predict_proba(X_val)
        score     = balanced_accuracy_score(y_val, np.argmax(val_proba, axis=1))
        fold_scores.append(score)
        print(f'  Fold {fold+1:2d}: BA={score:.4f}  best_iter={model.best_iteration_}')

        oof_proba[val_idx] += val_proba
        test_preds         += model.predict_proba(X_test_meta) / N_SPLITS

    oof_score = balanced_accuracy_score(y_train, np.argmax(oof_proba, axis=1))
    print(f'  OOF BA (seed={SEED}): {oof_score:.4f} | mean={np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')

    all_oof_proba  += oof_proba  / len(SEEDS)
    all_test_preds += test_preds / len(SEEDS)

final_oof = balanced_accuracy_score(y_train, np.argmax(all_oof_proba, axis=1))
print(f"\n{'='*60}")
print(f'FINAL OOF BA: {final_oof:.4f}   (a3 baseline: 0.3860)')
print(f"{'='*60}")


======================================== SEED=42 ========================================
  Fold  1: BA=0.3713  best_iter=24
  Fold  2: BA=0.4109  best_iter=87
  Fold  3: BA=0.3814  best_iter=40
  Fold  4: BA=0.3972  best_iter=1
  Fold  5: BA=0.3679  best_iter=35
  Fold  6: BA=0.4448  best_iter=287
  Fold  7: BA=0.3680  best_iter=105
  Fold  8: BA=0.3791  best_iter=17
  Fold  9: BA=0.3951  best_iter=24
  Fold 10: BA=0.4208  best_iter=27
  OOF BA (seed=42): 0.3933 | mean=0.3937 ± 0.0242

======================================== SEED=7 ========================================
  Fold  1: BA=0.3998  best_iter=55
  Fold  2: BA=0.4232  best_iter=35
  Fold  3: BA=0.3855  best_iter=45
  Fold  4: BA=0.4282  best_iter=127
  Fold  5: BA=0.3701  best_iter=16
  Fold  6: BA=0.4321  best_iter=39
  Fold  7: BA=0.3710  best_iter=12
  Fold  8: BA=0.3471  best_iter=9
  Fold  9: BA=0.4020  best_iter=223
  Fold 10: BA=0.3835  best_iter=24
  OOF BA (seed=7): 0.3946 | mean=0.3942 ± 0.0266

=================

In [8]:
# ── Cell 8: Per-class recall ─────────────────────────────────────────────────
oof_labels = np.argmax(all_oof_proba, axis=1)
report     = classification_report(y_train, oof_labels, output_dict=True)

disorder_names = {
    0:'레베르시', 1:'낭포성섬유증', 2:'당뇨', 3:'리증후군', 4:'암',
    5:'테이-삭스', 6:'혈색소침착증', 7:'사립체근병종', 8:'알츠하이머', 9:'확인안됨'
}
a3_recall = {0:0.314, 1:0.392, 2:0.272, 3:0.351, 4:0.759,
             5:0.338, 6:0.488, 7:0.237, 8:0.571, 9:0.138}

print(f'OOF BA: {final_oof:.4f}\n')
print(f'{"Class":<5} {"Name":<16} {"a3 recall":>10} {"Now":>8} {"Change":>8}')
print('-' * 55)
for cls in range(10):
    r     = report[str(cls)]['recall']
    r_old = a3_recall[cls]
    delta = r - r_old
    flag  = ' ← up' if delta > 0.02 else ' ← LOW' if r < 0.3 else ''
    print(f'{cls:<5} {disorder_names[cls]:<16} {r_old:>10.3f} {r:>8.3f} {delta:>+8.3f}{flag}')

OOF BA: 0.3838

Class Name              a3 recall      Now   Change
-------------------------------------------------------
0     레베르시                  0.314    0.321   +0.007
1     낭포성섬유증                0.392    0.403   +0.011
2     당뇨                    0.272    0.288   +0.016 ← LOW
3     리증후군                  0.351    0.369   +0.018
4     암                     0.759    0.690   -0.069
5     테이-삭스                 0.338    0.272   -0.066 ← LOW
6     혈색소침착증                0.488    0.494   +0.006
7     사립체근병종                0.237    0.338   +0.101 ← up
8     알츠하이머                 0.571    0.615   +0.044 ← up
9     확인안됨                  0.138    0.047   -0.091 ← LOW


In [9]:
# ── Cell 9: Save submission ──────────────────────────────────────────────────
final_preds = np.argmax(all_test_preds, axis=1)
submission  = pd.DataFrame({
    'id'      : TEST_DATA.index,
    'disorder': final_preds
}).set_index('id')
submission.to_csv('submission_etrf.csv')

print('Saved: submission_etrf.csv')
print(f'Shape: {submission.shape}')
print('\nPrediction distribution:')
print(submission['disorder'].value_counts().sort_index())
print(f'\nSubmit if OOF > 0.3860 (a3 baseline). Current OOF: {final_oof:.4f}')

Saved: submission_etrf.csv
Shape: (8834, 1)

Prediction distribution:
disorder
0     420
1    1504
2     662
3    1753
4     221
5    1067
6    1100
7    1647
8     232
9     228
Name: count, dtype: int64

Submit if OOF > 0.3860 (a3 baseline). Current OOF: 0.3838
